<h1>Graph Construction Using Actual distance based on raw miles</h1>

In [1]:
import geopandas as gpd
import pandas as pd

import numpy as np
import torch
from scipy.spatial.distance import cdist

<h1>Importing shapefile to use ZCTA

In [2]:
zip_gdf = gpd.read_file("../data/tl_2020_us_zcta520/tl_2020_us_zcta520.shp")

In [3]:
zip_gdf.head()

,ZCTA5CE20,GEOID20,CLASSFP20,MTFCC20,FUNCSTAT20,ALAND20,AWATER20,INTPTLAT20,INTPTLON20,geometry
0,35592,35592,B5,G6350,S,298552385,235989,+33.7427261,-088.0973903,"POLYGON ((-88.24735 33.6539, -88.24713 33.6541..."
1,35616,35616,B5,G6350,S,559506992,41870756,+34.7395036,-088.0193814,"POLYGON ((-88.13997 34.58184, -88.13995 34.582..."
2,35621,35621,B5,G6350,S,117838488,409438,+34.3350314,-086.7270557,"POLYGON ((-86.81659 34.3496, -86.81648 34.3496..."
3,35651,35651,B5,G6350,S,104521045,574316,+34.4609087,-087.4801507,"POLYGON ((-87.53087 34.42492, -87.53082 34.429..."
4,36010,36010,B5,G6350,S,335675180,236811,+31.6598950,-085.8128958,"POLYGON ((-85.95712 31.67744, -85.95676 31.677..."


In [4]:
print(zip_gdf.columns)

Index(['ZCTA5CE20', 'GEOID20', 'CLASSFP20', 'MTFCC20', 'FUNCSTAT20', 'ALAND20',
       'AWATER20', 'INTPTLAT20', 'INTPTLON20', 'geometry'],
      dtype='object')


In [5]:
df = pd.read_csv('../data/final_preprocessed_data_40_features.csv', usecols = ['ZIPCODE'])

zip_list = (
    df["ZIPCODE"]
    .astype(str)
    .unique()
    .tolist()
)

<h1>Filtering out the ZCTA based on the ZIPCODES that are present in our dataset 

In [6]:
zip_gdf = zip_gdf[zip_gdf["ZCTA5CE20"].isin(zip_list)].copy()
zip_gdf = zip_gdf.reset_index(drop=True)

In [ ]:
zip_gdf = zip_gdf.rename(columns={"ZCTA5CE20": "zip"})

In [ ]:
zip_gdf = zip_gdf.sort_values("zip").reset_index(drop=True)

In [10]:
print("Number of ZIPs:", len(zip_gdf))

Number of ZIPs: 197


In [11]:
print(zip_gdf.crs)

zip_gdf = zip_gdf.to_crs(epsg=4326)

EPSG:4269


In [12]:
zip_gdf["geometry"] = zip_gdf.geometry.buffer(0)

<h1>Getting a point inside zip polygon

In [13]:
zip_gdf["rep_point"] = zip_gdf.geometry.representative_point()

zip_gdf["lat"] = zip_gdf.rep_point.y
zip_gdf["lon"] = zip_gdf.rep_point.x

In [14]:
zip_gdf[["zip", "lat", "lon"]].head()

,zip,lat,lon
0,10001,40.751610,-74.004707
1,10002,40.716824,-73.985563
2,10003,40.731342,-73.990964
3,10004,40.688625,-74.018341
4,10005,40.706210,-74.008556


In [15]:
import osmnx as ox

G = ox.graph_from_place(
    "New York City, New York, USA",
    network_type="drive",
    simplify=True
)

G = ox.add_edge_speeds(G)
G = ox.add_edge_travel_times(G)

In [16]:
u, v, key = list(G.edges(keys=True))[0]
G.edges[u, v, key]

{'osmid': 25161349,
 'highway': 'motorway',
 'lanes': '2',
 'maxspeed': '50 mph',
 'name': 'Cross Island Parkway',
 'oneway': True,
 'ref': 'CI',
 'reversed': False,
 'length': np.float64(819.5016661477804),
 'geometry': <LINESTRING (-73.795 40.786, -73.795 40.786, -73.794 40.786, -73.794 40.786,...>,
 'speed_kph': 80.467,
 'travel_time': 36.66355149479923}

In [17]:
print("Number of road nodes:", len(G.nodes))
print("Number of road edges:", len(G.edges))

Number of road nodes: 55282
Number of road edges: 139192


In [18]:
zip_gdf["road_node"] = ox.nearest_nodes(
    G,
    zip_gdf["lon"].values,  # x = longitude
    zip_gdf["lat"].values   # y = latitude
)

In [19]:
print(zip_gdf[["zip", "road_node"]].head())

     zip   road_node
0  10001    42430600
1  10002    42432094
2  10003    42430872
3  10004  3659161914
4  10005  2824895141


In [20]:
import numpy as np
import networkx as nx

N = len(zip_gdf)
road_nodes = zip_gdf["road_node"].values

D = np.full((N, N), np.inf)

<h1>Finding shortest distances between zip nodes

In [21]:
for i in range(N):
    source_node = road_nodes[i]

    lengths = nx.single_source_dijkstra_path_length(
        G,
        source_node,
        weight="length"  
    )

    for j in range(N):
        target_node = road_nodes[j]
        if target_node in lengths:
            D[i, j] = lengths[target_node]

In [22]:
np.fill_diagonal(D, 0.0)

In [23]:
finite_vals = D[np.isfinite(D)]

print("Min distance (m):", finite_vals.min())
print("Max distance (m):", finite_vals.max())
print("Mean distance (m):", finite_vals.mean())

Min distance (m): 0.0
Max distance (m): 63904.20936561301
Mean distance (m): 18029.34851532811


In [24]:
import numpy as np

N = D.shape[0]
inf_mask = ~np.isfinite(D)
num_inf = inf_mask.sum()

print("Total entries:", N*N)
print("Inf entries:", num_inf, f"({100*num_inf/(N*N):.2f}%)")
print("Rows with any inf:", np.where(inf_mask.any(axis=1))[0].shape[0])

Total entries: 38809
Inf entries: 196 (0.51%)
Rows with any inf: 1


We can see there is one ZIP code that did not yield any direct path to other zip codes, reason for this could be: API error, dead end node and so on. 

In [25]:
import numpy as np

inf_mask = ~np.isfinite(D)

bad_rows = np.where(inf_mask.any(axis=1))[0]

bad_rows

array([95])

Imputing the bad zip

In [26]:
bad_row = bad_rows[0]

bad_zip_info = zip_gdf.loc[bad_row, ["zip", "lat", "lon", "road_node"]]
bad_zip_info

D[95, :] = D[:, 95]

In [ ]:
np.save('../data/distance_matrix_assym.npy', D)  

<h1>Making the graph symmetric

In [ ]:
D = 0.5 * (D + D.T)

In [ ]:
D

array([[    0.        ,  5390.51243316,  3330.22558045, ...,
        29519.68818022, 27113.25202834, 28435.59683894],
       [ 5390.51243316,     0.        ,  2283.6869125 , ...,
        24780.74772641, 23282.01836512, 24604.36317572],
       [ 3330.22558045,  2283.6869125 ,     0.        , ...,
        26412.86265957, 24171.82828872, 25494.17309933],
       ...,
       [29519.68818022, 24780.74772641, 26412.86265957, ...,
            0.        ,  4022.4884656 , 10808.19057637],
       [27113.25202834, 23282.01836512, 24171.82828872, ...,
         4022.4884656 ,     0.        ,  6876.89523412],
       [28435.59683894, 24604.36317572, 25494.17309933, ...,
        10808.19057637,  6876.89523412,     0.        ]], shape=(197, 197))

In [ ]:
np.save('distance_matrix.npy', D)  